In [ ]:
print("starting full run for Nelson")

!pip install torch torchvision -q

import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

print("torch imported")
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

DIM = 128
CLASSES = 100
BATCH = 64
EPOCHS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("building data transforms")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

print("loading CIFAR-100")
trainset = torchvision.datasets.CIFAR100('./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH, shuffle=True)

print("dataset ready:", len(trainset))
print("batches per epoch:", len(trainloader))

class VisualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128, DIM), nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class OGIFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(DIM * 2, 64), nn.ReLU(),
            nn.Linear(64, 2), nn.Softmax(dim=1)
        )
        self.gru = nn.GRUCell(DIM, DIM)
        self.cls = nn.Linear(DIM, CLASSES)

    def forward(self, vis, ling, h):
        w = self.attn(torch.cat([vis, ling], dim=1))
        h = self.gru(w[:, 0:1] * vis + w[:, 1:2] * ling, h)
        return h, self.cls(h), w

def softmax_att(Q, K, V):
    s = Q @ K.transpose(-2, -1) * (Q.shape[-1] ** -0.5)
    return torch.softmax(s, dim=-1) @ V

def linear_att(Q, K, V):
    phiQ = F.elu(Q) + 1
    phiK = F.elu(K) + 1
    KV = phiK.transpose(-2, -1) @ V
    Z = phiQ @ phiK.sum(dim=-2, keepdim=True).transpose(-2, -1)
    return (phiQ @ KV) / (Z + 1e-6)

class Observer(nn.Module):
    def __init__(self, heads=4):
        super().__init__()
        self.h = heads
        self.Wq = nn.Linear(DIM, DIM, bias=False)
        self.Wk = nn.Linear(DIM, DIM, bias=False)
        self.Wv = nn.Linear(DIM, DIM, bias=False)

    @torch.no_grad()
    def delta(self, vis, ling):
        x = torch.stack([vis, ling], dim=1)   # (B, 2, D)
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        B, S, D = Q.shape
        H = self.h
        dh = D // H

        Q = Q.view(B, S, H, dh).transpose(1, 2)  # (B, H, S, dh)
        K = K.view(B, S, H, dh).transpose(1, 2)
        V = V.view(B, S, H, dh).transpose(1, 2)

        out = []
        for b in range(B):
            s = softmax_att(Q[b], K[b], V[b]).reshape(H, -1)
            l = linear_att(Q[b], K[b], V[b]).reshape(H, -1)
            delta = 1 - F.cosine_similarity(s, l, dim=1)
            out.append(delta.detach().cpu().numpy())

        return np.mean(out, axis=0)  # (4,)

print("building model")

v = VisualEncoder().to(DEVICE)
t = nn.Embedding(CLASSES, DIM).to(DEVICE)
f = OGIFusion().to(DEVICE)
obs = Observer(4).to(DEVICE)

opt = optim.Adam(list(v.parameters()) + list(t.parameters()) + list(f.parameters()), lr=1e-3)
crit = nn.CrossEntropyLoss()

export = []
step = 0

print("starting training loop")

for epoch in range(EPOCHS):
    for batch_idx, (img, lbl) in enumerate(trainloader):
        img = img.to(DEVICE)
        lbl = lbl.to(DEVICE)

        lbl_c = lbl.clone()
        mask = torch.rand(lbl.size(0), device=DEVICE) > 0.5
        n = int(mask.sum().item())
        if n > 0:
            lbl_c[mask] = torch.randint(0, CLASSES, (n,), device=DEVICE)

        vis = v(img)
        ling = t(lbl_c)
        h = torch.zeros(img.size(0), DIM, device=DEVICE)

        h, logits, w = f(vis, ling, h)
        loss = crit(logits, lbl)

        opt.zero_grad()
        loss.backward()
        opt.step()

        delta = obs.delta(vis.detach(), ling.detach())

        export.append({
            "step_idx": step,
            "layer_idx": 0,
            "per_head_delta": [float(x) for x in delta.tolist()]
        })
        step += 1

        if batch_idx % 100 == 0:
            print("epoch", epoch + 1, "batch", batch_idx, "loss", float(loss.item()), "step", step)

print("saving file")

with open("ogi_navisad_export.json", "w") as fjson:
    json.dump(export, fjson)

print("done")
print("records:", len(export))
print("matches expected:", len(export) == len(trainloader))
print("first:", export[0] if export else "none")
print("last:", export[-1] if export else "none")